In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import evaluate
import torch

# Step 1: Load the FLAN-T5 model and tokenizer
model_name = "google/flan-t5-small"  # You can choose a larger model like flan-t5-base or flan-t5-large
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 2: Prepare a dataset (using CNN/DailyMail as an example)
dataset = load_dataset("cnn_dailymail", "3.0.0", split={"train": "train[:1%]", "test": "test[:1%]"})

def preprocess_data(batch):
    # Tokenize inputs and labels
    inputs = tokenizer(batch["article"], max_length=256, truncation=True, padding="max_length", return_tensors="pt")
    labels = tokenizer(batch["highlights"], max_length=64, truncation=True, padding="max_length", return_tensors="pt")
    inputs["labels"] = labels["input_ids"].masked_fill(labels["attention_mask"] == 0, -100)  # Ignore padding in loss calculation
    return inputs

# Apply preprocessing
dataset = dataset.map(preprocess_data, batched=True, remove_columns=["article", "highlights", "id"])

# Step 3: Prepare for fine-tuning
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# Define training arguments
training_args = TrainingArguments(
    output_dir="./flan-t5-summarization",  # Directory for saving checkpoints
    evaluation_strategy="steps",         # Evaluate at each step
    eval_steps=500,                       # Number of steps for evaluation
    save_steps=500,                       # Save the model checkpoint at each step
    per_device_train_batch_size=2,        # Reduced batch size to avoid OOM
    per_device_eval_batch_size=2,         # Reduced batch size for evaluation
    gradient_accumulation_steps=4,        # Simulate a larger batch size
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available()        # Enable mixed precision if using GPU
)

# Define a data collator
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Step 4: Fine-tune the model using Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=None,  # Deprecated, tokenizer removed
    data_collator=data_collator
)

# Start training
try:
    torch.cuda.empty_cache()  # Clear CUDA memory
    trainer.train()
except torch.cuda.OutOfMemoryError:
    print("Out of Memory! Try reducing batch size or sequence length further.")

# Step 5: Evaluate the model
# Use the evaluate library for metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

# Function to compute metrics
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE
    rouge_scores = rouge.compute(predictions=preds, references=labels)
    bleu_score = bleu.compute(predictions=preds, references=[[l] for l in labels])

    return {
        "rouge": rouge_scores,
        "bleu": bleu_score["bleu"]
    }

# Evaluate on the evaluation dataset
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shrihariharan1999 (shrihariharan1999-vellore-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss,Validation Loss
500,0.000000,nan
1000,0.000000,nan


ImportError: To be able to use evaluate-metric/rouge, you need to install the following dependencies['rouge_score'] using 'pip install rouge_score' for instance'